In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN attached:", True)
except Exception as e:
    print("HF_TOKEN missing (training continues, Hub push will skip):", e)


In [ ]:
!rm -rf /kaggle/working/arc && git clone --branch stage-a-cpt https://github.com/Nyvo2010/arc.git /kaggle/working/arc
!pip install -q -r /kaggle/working/arc/requirements-kaggle.txt huggingface_hub


In [ ]:
import glob, os, shutil
cand = glob.glob("/kaggle/input/*/jetmoe-8b/config.json")
if cand:
    base_src = os.path.dirname(os.path.dirname(cand[0]))
    print("reusing base weights from input:", base_src)
    shutil.copytree(base_src, "/kaggle/working/jetmoe-8b", dirs_exist_ok=True)
else:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="jetmoe/jetmoe-8b", local_dir="/kaggle/working/jetmoe-8b")
print("weights ready")


In [ ]:
import glob, os, shutil, json
restored_any = False
for d in sorted(glob.glob("/kaggle/input/*/checkpoints")):
    print("restoring checkpoints from input:", d)
    shutil.copytree(d, "/kaggle/working/checkpoints", dirs_exist_ok=True)
    restored_any = True
if not restored_any:
    print("no prior checkpoints found as kernel input; starting fresh")


In [ ]:
!cd /kaggle/working/arc && bash scripts/preflight_check.sh /kaggle/working/jetmoe-8b configs/phase1/tier_b.yaml


In [ ]:
!cd /kaggle/working/arc && python scripts/train_stage_a.py \
  --config configs/phase1/tier_b.yaml --variant layer_adaptive \
  --tier b --run-id phase1-tier-b-r1


In [ ]:
import json, glob
for f in sorted(glob.glob("/kaggle/working/checkpoints/phase1-tier-b/phase1-tier-b-r1/layer_adaptive/best.json")):
    d = json.load(open(f))
    print(f, "->", {k: d[k] for k in ("best_val_loss", "best_val_ppl", "step", "tokens_processed")})
st = json.load(open("/kaggle/working/checkpoints/phase1-tier-b/phase1-tier-b-r1/layer_adaptive/train_state.json"))
print("train_state: step", st.get("step"), "tokens", st.get("tokens_processed"))


In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN absent: skipping Hub push, keeping Kaggle outputs")
else:
    import subprocess
    rd = "/kaggle/working/checkpoints/phase1-tier-b/phase1-tier-b-r1/layer_adaptive"
    tag = "phase1-tierB-layer_adaptive-best"
    cmd = ("cd /kaggle/working/arc && python scripts/push_best_to_hub.py "
           "--repo-id Nyvo/arc-jetmoe-recurrence-phase1 --run-dir " + rd + " --tag " + tag)
    print("pushing", rd, "->", tag)
    r = subprocess.run(cmd, shell=True)
    print("push exit:", r.returncode)


In [ ]:
!ls -R /kaggle/working/checkpoints/phase1-tier-b/phase1-tier-b-r1/layer_adaptive 2>/dev/null | head -40
